# 03 — Two-Stage Default · REG · enet (Stage 2 conditional)

Stage 2 회귀 (y>0 only conditional, `E[Y|Y>0,x]`) ElasticNet — PP + X scaling + y target_transform + HP joint Optuna.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/reg/enet/`
- **PP**: PP joint Optuna (strategy_common §3 — ENet은 PP_FIXED 안 씀, 범위 탐색)
- **X scaling**: 5종 categorical (`StandardScaler / RobustScaler / YeoJohnson / Quantile / Hybrid`)
- **y target_transform**: 4종 categorical (`none / log1p / yeo-johnson / quantile`) — fold별 fit
- **Y_POSITIVE_ONLY**: True (y>0 die만 학습)
- **HPO**: N_TRIALS=1 (joint이라 비용 큼), anchor enqueue + Wide search
- **Sampler/Pruner/Timeout**: §4·§25


## 1. 환경 설정 + import

In [1]:
import os, sys, io, contextlib

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip = cleaning/outlier/scaling 등
GDRIVE_MODELING_ID     = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules
GDRIVE_OUTPUT_ID       = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip = 기존 실험 산출물 (RESUME용)
RESUME                 = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../../../../setup.py만
try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 + `from modules import ...` 가 3_modeling/modules를 찾게
PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer
from sklearn.model_selection import KFold

from scaling import HybridScaler       # skew 기반 하이브리드 스케일러 (2_preprocessing/scaling.py)
from meta_features import add_meta_features

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

setup 완료
PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
optuna v4.7.0


## 2. 실험 설정

In [ ]:
# Two-Stage Stage 2 고정: ElasticNet, y>0 die만 학습 (= E[Y|Y>0,x])
REG_MODEL_NAME = 'enet'
EXP_ID   = f'ts-reg-{REG_MODEL_NAME}-001'  # study_name + 산출물 폴더 접두어
EXP_MEMO = 'Two-Stage default · REG · enet · y>0 conditional · PP+scaling+y_transform joint'
USER     = 'jh'  # DB 파일명 접두어

N_TRIALS         = 3000   # PP/scaling/y변환/HP joint 탐색이라 trial 많이 할당
N_FOLDS          = 5      # OOF RMSE 안정성 vs 연산 비용 균형
N_STARTUP_TRIALS = 50     # TPE warm-up: 이 수만큼 무작위 탐색 후 베이즈 학습 시작
N_JOBS           = 5      # ElasticNet 자체는 n_jobs 인자 없음 — 표기만
TIMEOUT_SEC      = 20 * 60 * 60  # 초 단위 안전망 (None=무제한, Colab 타임아웃 대비)

CLIP_Y_EXTREME  = True   # train y의 max(1.0, 1건)를 두 번째 큰 값으로 clip (학습 안정화)
Y_POSITIVE_ONLY = True   # Stage 2: y>0 die만으로 학습 (= E[Y|Y>0,x])

# 산출물 경로 — 실험 번호(001,002,...) 단위로 분리하여 재실험 시 이전 결과 보존
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'reg', REG_MODEL_NAME, 'hp', EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')  # study DB (RESUME 시 이어서 탐색)
os.makedirs(OUT_DIR, exist_ok=True)

# anchor: 1차 reg-enet best HP → trial 0으로 enqueue하여 warm-start
# 전처리/스케일/y변환/HP를 하나의 dict에 (joint Optuna 탐색과 동일 키 구조)
ENET_ANCHOR = {
    'missing_threshold':          0.5,    # 결측률 ≥ 이 값이면 feature 제거
    'corr_threshold':             0.90,   # 피처간 |r| ≥ 이 값이면 한쪽 제거 (다중공선성)
    'add_indicator':              True,   # 결측 여부 indicator 컬럼 추가 여부
    'indicator_threshold':        0.10,   # 결측률 ≥ 이 값일 때만 indicator 추가
    'spatial_max_dist':           5.0,    # 공간 imputation 최대 탐색 거리
    'post_impute_corr_threshold': 0.97,   # imputation 후 고상관 제거 임계값
    'scaling':                    'RobustScaler',  # X 스케일러 (ElasticNet은 스케일 의존)
    'target_transform':           'log1p',         # y 변환 (fold별 fit으로 leakage 방지)
    'alpha':                      0.000145,        # ElasticNet 정규화 강도
    'l1_ratio':                   0.725,           # L1 비중 (0=Ridge, 1=Lasso)
    'max_iter':                   15000,           # 수렴 최대 반복 수
}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS} | TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'Y_POSITIVE_ONLY={Y_POSITIVE_ONLY}')
print(f'OUT_DIR={OUT_DIR}')

## 3. 데이터 로드 + Y clip

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# y 변환은 글로벌로 안 정하고 trial마다 + fold마다 fit (leakage 방지) — 후보 3종(none / log1p / quantile)을 objective에서 categorical로 탐색 (yeo-johnson은 제외)
print('[target transform] Optuna가 trial별 자동 탐색 (none/log1p/yeo-johnson/quantile)')

[load_xs] all-NaN 행 407개 제거 → 174,573행


[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572


[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729


xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[target transform] Optuna가 trial별 자동 탐색 (none/log1p/yeo-johnson/quantile)


## 4. KFold split + helper (scaling, target transform, PP)

- KFold는 unit-level (모든 trial 공유)
- `make_target_transformer`: fold별 train fold y에 fit (leakage 방지)
- ENet은 inverse 후 음수 가능 → `np.clip(0, None)`

In [4]:
# unit ID 단위 KFold — 모든 trial이 공유
unit_ids_train = ys_input['train'][KEY_COL].unique()
_kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = []
for tr_idx, vl_idx in _kf.split(unit_ids_train):
    FOLDS.append((unit_ids_train[tr_idx], unit_ids_train[vl_idx]))
print(f'[KFold] {N_FOLDS} folds, unit 단위 분할')


def make_scaler(name):
    # 이름 → sklearn 스케일러 객체
    if name == 'StandardScaler':
        return StandardScaler()
    if name == 'RobustScaler':
        return RobustScaler()
    if name == 'YeoJohnson':
        return PowerTransformer(method='yeo-johnson', standardize=True)
    if name == 'Quantile':
        return QuantileTransformer(output_distribution='normal', random_state=SEED)
    if name == 'Hybrid':
        return HybridScaler(skew_threshold=10.0)
    raise ValueError(f'Unknown scaling: {name!r}')


def make_target_transformer(name, y_train_arr):
    """y 변환 이름 → (forward_fn, inverse_fn). 변환기가 필요한 종류는 train fold y에만 fit (leakage 방지).

    선형 모델 예측은 inverse 후 음수 가능 → 모든 inverse는 마지막에 np.clip(0, None).
    후보: 'none' / 'log1p' / 'yeo-johnson'(PowerTransformer) / 'quantile'(rank→normal)
    """
    if name == 'none':
        return (
            lambda y: np.asarray(y, dtype=float),
            lambda y: np.clip(np.asarray(y, dtype=float), 0.0, None),
        )
    if name == 'log1p':
        return (
            lambda y: np.log1p(np.asarray(y, dtype=float)),
            lambda y: np.clip(np.expm1(np.asarray(y, dtype=float)), 0.0, None),
        )
    if name == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson', standardize=False)
        pt.fit(np.asarray(y_train_arr, dtype=float).reshape(-1, 1))
        return (
            lambda y: pt.transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(),
            lambda y: np.clip(pt.inverse_transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(), 0.0, None),
        )
    if name == 'quantile':
        n_q = min(1000, len(y_train_arr))
        qt = QuantileTransformer(output_distribution='normal', n_quantiles=n_q, random_state=SEED)
        qt.fit(np.asarray(y_train_arr, dtype=float).reshape(-1, 1))
        return (
            lambda y: qt.transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(),
            lambda y: np.clip(qt.inverse_transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(), 0.0, None),
        )
    raise ValueError(f'Unknown target_transform: {name!r}')


def run_pp_silent(pp_params):
    """preprocess.run을 호출하되 stdout를 삼킴. corr_keep_by 류는 'std' 강제 (leakage 방지 — target 보고 feature 고르면 OOF 편향)."""
    pp_full = dict(pp_params, corr_keep_by='std', post_impute_corr_keep_by='std')
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        out = preprocess.run(xs.copy(), ys_input, feat_cols, xs_dict, params=pp_full)
    return out


print('[helper] make_scaler / make_target_transformer / run_pp_silent 정의 완료')

[KFold] 5 folds, unit 단위 분할
[helper] make_scaler / make_target_transformer / run_pp_silent 정의 완료


## 5. Optuna joint HPO (PP + scaling + target_transform + enet HP)

- objective: y>0 die만 fit, OOF unit RMSE
- anchor enqueue (1차 best HP 시작점)

In [ ]:
y_train_unit_df = ys_input['train']
y_train_unit    = y_train_unit_df.set_index(KEY_COL)[TARGET_COL]   # unit-level 정답


def _broadcast_y_to_die(xs_split, y_unit_series):
    # unit 레이블을 die 행에 확산 (같은 unit의 4 die에 같은 y 값 복사)
    return xs_split[KEY_COL].map(y_unit_series).values.astype(float)


def _aggregate_die_to_unit_mean(xs_split, die_pred):
    # die 예측 → unit 평균 집계 (Stage 2 단독 평가용)
    df = pd.DataFrame({KEY_COL: xs_split[KEY_COL].values, 'pred': die_pred})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()


def objective(trial):
    # 한 trial = 전처리 6축 + 스케일링 1축 + y변환 1축 + ElasticNet HP 3개를 한꺼번에 샘플

    pp_params = {
        'missing_threshold':          trial.suggest_float('missing_threshold', 0.30, 0.90),
        'corr_threshold':             trial.suggest_float('corr_threshold', 0.88, 0.98),
        'add_indicator':              trial.suggest_categorical('add_indicator', [True, False]),
        'indicator_threshold':        trial.suggest_float('indicator_threshold', 0.05, 0.20),
        'spatial_max_dist':           trial.suggest_float('spatial_max_dist', 1.0, 6.0),
        'post_impute_corr_threshold': trial.suggest_float('post_impute_corr_threshold', 0.96, 0.99),
    }
    scaling_name = trial.suggest_categorical(
        'scaling', ['StandardScaler', 'RobustScaler', 'YeoJohnson', 'Quantile', 'Hybrid']
    )
    # yeo-johnson은 후보 제외 — 작은 alpha 조합에서 inverse가 깨져 NaN 발생 이력
    target_transform_name = trial.suggest_categorical(
        'target_transform', ['none', 'log1p', 'quantile']
    )
    enet_hp = dict(
        alpha=trial.suggest_float('alpha', 1e-7, 1e-3, log=True),
        l1_ratio=trial.suggest_float('l1_ratio', 0.50, 0.95),  # 0.95까지 (순수 Lasso는 약신호를 다 0으로 처리해 위험)
        max_iter=trial.suggest_int('max_iter', 8000, 20000, step=1000),
        tol=1e-6, selection='random', precompute=True, random_state=SEED,
    )

    # 전처리 (조용히 — stdout은 버림, 실패하면 trial만 prune)
    try:
        out = run_pp_silent(pp_params)
    except Exception as e:
        raise optuna.exceptions.TrialPruned(f'PP failed: {e}')
    xs_train_c      = out['xs_train']
    feat_cols_clean = out['feat_cols']

    # ElasticNet용 메타피처: position OHE + 위치 기반 X OHE (die_xy 제외 — 선형에 무의미)
    feat_cols_clean = add_meta_features(
        xs_train_c, None, None, feat_cols_clean,
        position_mode='ohe', use_die_xy=False, verbose=False,
        use_loc_x_ohe=True,
        loc_x_required=("X1073",),
        loc_x_optional=("X1059", "X1075", "X1076", "X1077"),
    )

    y_die_orig = _broadcast_y_to_die(xs_train_c, y_train_unit)

    n_tr = len(xs_train_c)
    oof = np.full(n_tr, np.nan)

    for tr_units, vl_units in FOLDS:
        tr_mask = xs_train_c[KEY_COL].isin(set(tr_units)).values
        vl_mask = xs_train_c[KEY_COL].isin(set(vl_units)).values

        X_tr = xs_train_c.loc[tr_mask, feat_cols_clean].values
        X_vl = xs_train_c.loc[vl_mask, feat_cols_clean].values

        # Stage 2 핵심: fit은 y>0 die만으로 (= E[Y|Y>0,x]). 예측(OOF)은 val fold 전체 die에 함
        y_tr_orig = y_die_orig[tr_mask]
        if Y_POSITIVE_ONLY:
            pos_mask = y_tr_orig > 0
            X_tr_fit = X_tr[pos_mask]
            y_tr_fit = y_tr_orig[pos_mask]
        else:
            X_tr_fit = X_tr
            y_tr_fit = y_tr_orig

        # y변환기: 그 fold의 학습 y(>0)에만 fit (leakage 방지 — val fold y는 보지 않음)
        forward_fn, inverse_fn = make_target_transformer(target_transform_name, y_tr_fit)
        y_fit = forward_fn(y_tr_fit)

        # 스케일러: 학습 X(y>0 die)에 fit → val X에 transform만 (fit 금지)
        scaler = make_scaler(scaling_name)
        X_tr_s = scaler.fit_transform(X_tr_fit)
        X_vl_s = scaler.transform(X_vl)

        try:
            model = ElasticNet(**enet_hp)
            model.fit(X_tr_s, y_fit)
            pred_t = model.predict(X_vl_s)
        except Exception as e:
            raise optuna.exceptions.TrialPruned(f'enet fit/predict failed: {e}')
        pred_inv = inverse_fn(pred_t)
        # NaN/Inf 방어: 특정 y변환+스케일 조합에서 역변환 폭발 가능 → 이 trial만 prune
        if not np.isfinite(pred_inv).all():
            raise optuna.exceptions.TrialPruned(
                f'NaN/Inf in inverse pred | '
                f'pred_NaN={int(np.isnan(pred_t).sum())}, pred_inf={int(np.isinf(pred_t).sum())}, '
                f'inv_NaN={int(np.isnan(pred_inv).sum())}, inv_inf={int(np.isinf(pred_inv).sum())} | '
                f'tt={target_transform_name}, sc={scaling_name}, '
                f'alpha={enet_hp["alpha"]:.2e}, l1r={enet_hp["l1_ratio"]:.3f}'
            )
        oof[vl_mask] = pred_inv

    # fold 누락 방어 (prune로 안 잡혔다면 마스크 버그)
    if not np.isfinite(oof).all():
        n_nan = int(np.isnan(oof).sum())
        n_inf = int(np.isinf(oof).sum())
        raise RuntimeError(
            f'OOF has non-finite values (NaN={n_nan}, Inf={n_inf}) — fold coverage bug'
        )

    # objective = mean 집계 unit OOF RMSE (Stage 2 단독 — ×P(Y>0)는 combine에서)
    unit_pred = _aggregate_die_to_unit_mean(xs_train_c, oof)
    aligned   = unit_pred.set_index(KEY_COL)['pred'].loc[y_train_unit.index]
    train_rmse = float(np.sqrt(np.mean((aligned.values - y_train_unit.values) ** 2)))
    if not np.isfinite(train_rmse):
        raise optuna.exceptions.TrialPruned(
            f'train_rmse non-finite ({train_rmse}) | tt={target_transform_name}, sc={scaling_name}'
        )
    # user_attr: study DB에 영구 저장 — 후분석 및 best trial 재현용
    trial.set_user_attr('train_rmse', train_rmse)
    trial.set_user_attr('n_features_after_pp', len(feat_cols_clean))
    return train_rmse


# TPE: multivariate=HP 결합 분포, group=조건부 축 skip, seed=None → run마다 다양성
# load_if_exists=RESUME: True이면 기존 DB 이어서, False이면 신규 (DB 있으면 DuplicatedStudyError)
study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=RESUME,
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
    pruner=MedianPruner(n_warmup_steps=10),  # 10 step warm-up 후 중앙값 대비 나쁜 trial 가지치기
)
# study_meta 박제 — 어떤 조건으로 학습됐는지 DB만 보고도 알 수 있게
study.set_user_attr('exp_id', EXP_ID)
study.set_user_attr('exp_memo', EXP_MEMO)
study.set_user_attr('user', USER)
study.set_user_attr('y_positive_only', Y_POSITIVE_ONLY)
study.set_user_attr('anchor', ENET_ANCHOR)
if len(study.trials) == 0:
    study.enqueue_trial(ENET_ANCHOR)  # anchor를 trial 0으로 주입 → warm-start

study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

best_params = dict(study.best_trial.params)
best_value  = float(study.best_value)
print(f'\n[HPO 완료] best OOF RMSE = {best_value:.6f}')
print(f'best_params = {best_params}')

## 6. Best trial refit (5-fold + die-level pred 캐쳐)

In [ ]:
# best_params에서 전처리/스케일/y변환/ElasticNet HP 분리
PP_KEYS = ('missing_threshold', 'corr_threshold', 'add_indicator',
           'indicator_threshold', 'spatial_max_dist', 'post_impute_corr_threshold')
best_pp               = {k: best_params[k] for k in PP_KEYS if k in best_params}
best_scaling          = best_params.get('scaling', 'RobustScaler')
best_target_transform = best_params.get('target_transform', 'log1p')
best_enet             = {k: best_params[k] for k in ('alpha', 'l1_ratio', 'max_iter')}
best_enet.update(tol=1e-6, selection='random', precompute=True, random_state=SEED)

# best 전처리 조합으로 다시 한 번 (로그 출력 살림) → 3 split 전처리된 DataFrame 확보
pp_full = dict(best_pp, corr_keep_by='std', post_impute_corr_keep_by='std')
out = preprocess.run(xs.copy(), ys_input, feat_cols, xs_dict, params=pp_full)
xs_train_c      = out['xs_train']
xs_val_c        = out['xs_val']
xs_test_c       = out['xs_test']
feat_cols_clean = out['feat_cols']

# 메타피처 (objective와 동일)
feat_cols_clean = add_meta_features(
    xs_train_c, xs_val_c, xs_test_c, feat_cols_clean,
    position_mode='ohe', use_die_xy=False,
    use_loc_x_ohe=True,
    loc_x_required=("X1073",),
    loc_x_optional=("X1059", "X1075", "X1076", "X1077"),
)

y_die_orig = _broadcast_y_to_die(xs_train_c, y_train_unit)

n_tr, n_vl, n_te = len(xs_train_c), len(xs_val_c), len(xs_test_c)
oof_pred  = np.full(n_tr, np.nan)
val_pred  = np.zeros(n_vl)
test_pred = np.zeros(n_te)

fold_models  = []
fold_scalers = []

for i, (tr_units, vl_units) in enumerate(FOLDS):
    tr_mask = xs_train_c[KEY_COL].isin(set(tr_units)).values
    vl_mask = xs_train_c[KEY_COL].isin(set(vl_units)).values

    X_tr = xs_train_c.loc[tr_mask, feat_cols_clean].values
    X_vl = xs_train_c.loc[vl_mask, feat_cols_clean].values
    X_v  = xs_val_c[feat_cols_clean].values
    X_te = xs_test_c[feat_cols_clean].values

    # objective와 동일하게 y>0 die만 fit
    y_tr_orig = y_die_orig[tr_mask]
    if Y_POSITIVE_ONLY:
        pos_mask = y_tr_orig > 0
        X_tr_fit = X_tr[pos_mask]
        y_tr_fit = y_tr_orig[pos_mask]
    else:
        X_tr_fit = X_tr
        y_tr_fit = y_tr_orig

    forward_fn, inverse_fn = make_target_transformer(best_target_transform, y_tr_fit)
    y_fit = forward_fn(y_tr_fit)

    scaler = make_scaler(best_scaling)
    X_tr_s = scaler.fit_transform(X_tr_fit)
    X_vl_s = scaler.transform(X_vl)
    X_v_s  = scaler.transform(X_v)
    X_te_s = scaler.transform(X_te)

    model = ElasticNet(**best_enet)
    model.fit(X_tr_s, y_fit)

    # 예측은 변환 공간 → inverse. val/test는 fold 평균 누적
    oof_pred[vl_mask] = inverse_fn(model.predict(X_vl_s))
    val_pred  += inverse_fn(model.predict(X_v_s))  / N_FOLDS
    test_pred += inverse_fn(model.predict(X_te_s)) / N_FOLDS

    fold_models.append(model)
    fold_scalers.append(scaler)
    print(f'[refit fold {i+1}/{N_FOLDS}] tr_units={len(tr_units)}, vl_units={len(vl_units)}, fit_n={len(X_tr_fit):,}')

# finite 가드 — HPO에서 NaN trial을 prune했다면 여기 도달 안 함
for _name, _arr in (('oof_pred', oof_pred), ('val_pred', val_pred), ('test_pred', test_pred)):
    if not np.isfinite(_arr).all():
        _n_nan = int(np.isnan(_arr).sum())
        _n_inf = int(np.isinf(_arr).sum())
        raise RuntimeError(
            f'[refit] {_name} has non-finite values (NaN={_n_nan}, Inf={_n_inf}) — '
            f'best_params={best_target_transform}/{best_scaling} 조합 점검 필요'
        )

# mean 집계 unit 예측 + RMSE (Stage 2 단독이라 RMSE 자체는 의미 제한적 — 진짜 평가는 combine에서)
oof_unit  = _aggregate_die_to_unit_mean(xs_train_c, oof_pred)
val_unit  = _aggregate_die_to_unit_mean(xs_val_c,   val_pred)
test_unit = _aggregate_die_to_unit_mean(xs_test_c,  test_pred)

y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

oof_rmse  = float(np.sqrt(np.mean((oof_unit.set_index(KEY_COL)['pred'].loc[y_train_unit.index].values - y_train_unit.values)**2)))
val_rmse  = float(np.sqrt(np.mean((val_unit.set_index(KEY_COL)['pred'].loc[y_val_true.index].values  - y_val_true.values)**2)))
test_rmse = float(np.sqrt(np.mean((test_unit.set_index(KEY_COL)['pred'].loc[y_test_true.index].values - y_test_true.values)**2)))

print(f'\n[Refit 완료] (reg 단독, y>0 conditional)')
print(f'  best target_transform = {best_target_transform}')
print(f'  best scaling          = {best_scaling}')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

## 7. 산출물 저장

In [7]:
# hpo.save_artifacts가 받는 dict 형식으로 포장 (enet은 π/μ 없음 → None)
refit_result = {
    'oof_pred_die':         oof_pred,
    'val_pred_die':         val_pred,
    'test_pred_die':        test_pred,
    'oof_pi':  None, 'val_pi':  None, 'test_pi':  None,
    'oof_mu':  None, 'val_mu':  None, 'test_mu':  None,
    'oof_pred_unit':        oof_unit,
    'val_pred_unit':        val_unit,
    'test_pred_unit':       test_unit,
    'fold_models':          fold_models,
    'fold_scalers':         fold_scalers,
    'best_params_resolved': {**best_pp,
                             'scaling':          best_scaling,
                             'target_transform': best_target_transform,
                             **best_enet},
    'model_name':           REG_MODEL_NAME,
}

study_meta_for_save = {
    'exp_id':                EXP_ID,
    'exp_memo':              EXP_MEMO,
    'user':                  USER,
    'model_name':            REG_MODEL_NAME,
    'best_target_transform': best_target_transform,
    'best_scaling':          best_scaling,
    'best_pp':               best_pp,
    'y_positive_only':       Y_POSITIVE_ONLY,
    'clip_y_extreme':        CLIP_Y_EXTREME,
    'n_trials':              N_TRIALS,
    'n_folds':               N_FOLDS,
    'n_jobs':                N_JOBS,
    'timeout_sec':           TIMEOUT_SEC,
    'seed_kfold':            SEED,
    'anchor':                ENET_ANCHOR,
    'hpo_best_value':        best_value,
}

# die/unit CSV 6개 + fold_models.pkl + best_params.json 저장. postprocess_config=None → mean 집계 (후처리는 combine에서)
hpo.save_artifacts(
    refit_result=refit_result,
    xs_train=xs_train_c, xs_val=xs_val_c, xs_test=xs_test_c,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=None,
    study_meta=study_meta_for_save,
)

for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'reg_{REG_MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크')
        display(FileLink(_zip))
except ImportError:
    pass

[save_artifacts] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\reg\enet 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=mean)
  best_params.json                       9.8 KB
  fold_models.pkl                       77.7 KB
  oof_die.csv                        5,504.3 KB
  oof_unit.csv                         950.2 KB
  optuna_jh_ts-reg-enet-002.db         112.0 KB
  test_die.csv                       1,834.3 KB
  test_unit.csv                        316.7 KB
  val_die.csv                        1,834.8 KB
  val_unit.csv                         316.7 KB
